# Permuting and sampling data to calculate the null hypothesis

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [ ]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.csv') and ('null-' not in f)]
print(files)

In [ ]:
null_output_file = 'null-corpora/{}-null-corpus.parquet'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

## Setting up the null-test data

### Load, permute, save

In [ ]:
acceptable_columns = ['turn_delta', 'self',
                     'x_speaker',
                     'dyad', 'corpus', 'conversation_id', 'x_turn_id',
                      'x_utterance', 'y_utterance'
                      ]

In [ ]:
df = []

In [ ]:
for f in tqdm(files):
    print(f)
    df += [pd.read_csv(f)]
    df[-1] = df[-1].loc[
        (~df[-1]['x_utterance'].isna())
        & (~df[-1]['y_utterance'].isna())
    ]
    df[-1]['turn_delta'] = df[-1]['y_turn_id'] - df[-1]['x_turn_id']
    df[-1]['dyad'] = df[-1]['x_speaker'].astype(str) + '-' + df[-1]['y_speaker'].astype(str)


    selx = df[-1]['x_utterance'].apply(lambda x: len(re.findall(r'\s+', x)) >= 1)
    sely = df[-1]['y_utterance'].apply(lambda x: len(re.findall(r'\s+', x)) >= 1)
    df[-1] = df[-1].loc[selx & sely]
    df[-1] = df[-1][acceptable_columns]

df = pd.concat(df,ignore_index=True)
df.shape

In [ ]:
df = df.loc[(df['x_turn_id'] >= 5)]
df.shape

In [ ]:
df.head()

In [ ]:
df['conversation_id'] = ['-'.join(x.split('-')[:-1]) for x in tqdm(df['x_speaker'])]

In [ ]:
df[['x_utterance', 'y_utterance']].isna().mean()

#### Permuting the utterances

In [ ]:
df['null_y_utterance'] = df['y_utterance'].sample(frac=1).values
df['bad'] = df['null_y_utterance'] == df['y_utterance']
df['bad'].sum(), df['bad'].mean()

In [ ]:
del df['y_utterance']
del df['bad']
df = df.rename(columns={'null_y_utterance': 'y_utterance'})

#### Save a checkpoint

In [ ]:
df.to_parquet('ckpt.parquet')

### Loading from a checkpoint

In [ ]:
df = pd.read_parquet('ckpt.parquet')

### Split the data into a k-fold, 10% split

In [ ]:
df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1

In [ ]:
df['x_turn_id'] = df['conversation_id'].astype(str) + df['x_turn_id'].astype(str)

In [ ]:
df = df.drop_duplicates(subset=['corpus', 'x_turn_id', 'self'], keep='last')

In [ ]:
good_examples = pd.merge(
    left=df.loc[df['self']], left_on='x_turn_id',
    right=df[['x_turn_id', 'self', 'sample_delta']].loc[~df['self']], right_on='x_turn_id',
    how='left'
)

good_examples = good_examples.loc[
    ((good_examples['sample_delta_x'] >= 3)
    & (good_examples['sample_delta_x'] <= 900))
    & ((good_examples['sample_delta_y'] >= 3)
    & (good_examples['sample_delta_y'] <= 900))
]

In [ ]:
print(df.shape)
df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
df.shape

In [ ]:
# sample_ids = []

In [ ]:
# for corpus in tqdm(df['corpus'].unique()):
#     # print(corpus)
#     sub = df['x_turn_id'].loc[df['corpus'].isin([corpus])].unique()
#     k = int(np.ceil(len(sub) * .9))
#     sample_ids += np.random.choice(sub, size=(k,), replace=False).tolist()
# len(sample_ids)

In [ ]:
sid_doc = pd.DataFrame()
sid_doc['x_turn_id'] = df['x_turn_id'].values
sid_doc.to_csv('x_turn_id-sample.csv', index=False, encoding='utf-8')

## Creating final document

In [ ]:
df = pd.read_parquet('ckpt.parquet')
xids = pd.read_csv('x_turn_id-sample.csv')['x_turn_id'].values

In [ ]:
df['x_turn_id_'] = df['conversation_id'].astype(str) + df['x_turn_id'].astype(str)

In [ ]:
df = df.loc[df['x_turn_id_'].isin(xids)]
df.shape

In [ ]:
df.head()

### Final checks and outputs

In [ ]:
# df['x_turn_id'].loc[df['y_utterance'].isna()].nunique()

In [ ]:
null_output_file = os.path.join(data_loc,'null-corpus.parquet')

In [ ]:
df.to_parquet(null_output_file)

#### Test that stitched doc is okay

In [ ]:
null_output_file = os.path.join(data_loc,'null-corpus.parquet')

In [ ]:
df = pd.read_parquet(null_output_file)
# df = pd.read_csv(null_output_file)
# df = pd.read_table(null_output_file, sep='\t')
df.shape

In [ ]:
df[['x_utterance', 'y_utterance']].isna().sum()